In [2]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from pathlib import Path
from scipy.fftpack import dct
import pywt

os.makedirs("plots", exist_ok=True)
sns.set_theme(style="whitegrid")
sns.set_context("talk")

plt.rcParams.update({
    'figure.figsize': (10, 7), 
    'font.size': 24,
    'axes.titlesize': 20,
    'axes.labelsize': 20,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'axes.grid': True,
    'grid.linestyle': '--',
    'grid.alpha': 0.6
})

datasets = ['Kodak', 'McM', 'SIPI']
dataset_dirs = {
    'Kodak': 'kodak_dataset',
    'McM': 'McM_dataset',
    'SIPI': 'sipi_dataset',
}

In [ ]:
# ensure the image is grayscale and in [0, 1] range
def ensure_grayscale_float(image: np.ndarray) -> np.ndarray:
    if image.ndim == 3:
        image = image[..., :3].mean(axis=2)
    image = image.astype(np.float64)
    if image.max() > 1.0:
        image = image / 255.0
    return image

# gets the DCT coefficients of the image in block-wise manner 
def block_dct_coeffs(image: np.ndarray, block: int = 8) -> np.ndarray:
    h, w = image.shape
    pad_h = (block - (h % block)) % block
    pad_w = (block - (w % block)) % block
    padded = np.pad(image, ((0, pad_h), (0, pad_w)), mode='edge')

    coeff_list = []
    for r in range(0, padded.shape[0], block):
        for c in range(0, padded.shape[1], block):
            blk = padded[r:r + block, c:c + block]
            c2 = dct(dct(blk.T, norm='ortho').T, norm='ortho')
            coeff_list.append(c2.ravel())
    return np.concatenate(coeff_list)

# gets the DWT coefficients of the image using the specified wavelet and decomposition level
def dwt_coeffs(image: np.ndarray, wavelet: str = 'bior4.4', level: int = 3) -> np.ndarray:
    coeffs = pywt.wavedec2(image, wavelet=wavelet, level=level)
    flat = [coeffs[0].ravel()]
    for detail_level in coeffs[1:]:
        for band in detail_level:
            flat.append(band.ravel())
    return np.concatenate(flat)

# computes the cumulative energy curve from the given coefficients, which represents the proportion of total energy captured by the top coefficients
def cumulative_energy_curve(coeffs: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    energy = np.square(np.abs(coeffs)).astype(np.float64)
    energy.sort()
    energy = energy[::-1]
    
    total = energy.sum()
    if total <= 0:
        x = np.linspace(0.0, 1.0, 100)
        return x, np.zeros_like(x)
    cum = np.cumsum(energy) / total
    x = np.arange(1, len(cum) + 1) / len(cum)
    return x, cum

# used during energy compaction computation 
def pick_sample_image(dataset_name: str) -> Path | None:
    folder = Path(dataset_dirs[dataset_name])
    if not folder.exists():
        return None
    candidates = sorted([p for p in folder.iterdir() if p.is_file()])
    return candidates[0] if candidates else None


In [ ]:
for dataset_name in datasets:
    try:
        jpeg_df = pd.read_csv(f"{dataset_name}_jpeg_avg_results.csv").sort_values('CR')
        j2k_df = pd.read_csv(f"{dataset_name}_jpeg2000_avg_results.csv").sort_values('CR')
    except Exception:
        continue

    # 1. CR vs Quality (PSNR)
    plt.figure()
    plt.plot(jpeg_df['CR'], jpeg_df['PSNR'], marker='o', label='JPEG (DCT)', linewidth=2)
    plt.plot(j2k_df['CR'], j2k_df['PSNR'], marker='s', label='JPEG2000 (DWT)', linewidth=2)
    plt.title(f'Compression Ratio vs Quality (PSNR) - {dataset_name}')
    plt.xlabel('Compression Ratio (CR)')
    plt.ylabel('PSNR (dB)')
    plt.legend()
    plt.grid(True, alpha=0.35)
    plt.tight_layout()
    plt.savefig(f"plots/{dataset_name}_cr_vs_psnr.png", dpi=150)
    plt.close()

    # 2. Bitrate vs Distortion (BPP vs PSNR)
    plt.figure()
    jpeg_bpp = jpeg_df.sort_values('BPP')
    j2k_bpp = j2k_df.sort_values('BPP')
    plt.plot(jpeg_bpp['BPP'], jpeg_bpp['PSNR'], marker='o', label='JPEG (DCT)', linewidth=2)
    plt.plot(j2k_bpp['BPP'], j2k_bpp['PSNR'], marker='s', label='JPEG2000 (DWT)', linewidth=2)
    plt.title(f'Bitrate vs Distortion - {dataset_name}')
    plt.xlabel('Bits Per Pixel (BPP)')
    plt.ylabel('PSNR (dB)')
    plt.legend()
    plt.grid(True, alpha=0.35)
    plt.xlim(max(jpeg_df['BPP'].max(), j2k_df['BPP'].max()) + 0.2, 0)
    plt.tight_layout()
    plt.savefig(f"plots/{dataset_name}_bpp_vs_psnr.png", dpi=150)
    plt.close()

    # 2b. Bitrate vs Distortion (MSE)
    plt.figure()
    plt.plot(jpeg_bpp['BPP'], jpeg_bpp['MSE'], marker='o', label='JPEG (DCT)', linewidth=2)
    plt.plot(j2k_bpp['BPP'], j2k_bpp['MSE'], marker='s', label='JPEG2000 (DWT)', linewidth=2)
    plt.title(f'Bitrate vs Distortion (MSE) - {dataset_name}')
    plt.xlabel('Bits Per Pixel (BPP)')
    plt.ylabel('MSE')
    plt.legend()
    plt.grid(True, alpha=0.35)
    plt.xlim(max(jpeg_df['BPP'].max(), j2k_df['BPP'].max()) + 0.2, 0)
    plt.tight_layout()
    plt.savefig(f"plots/{dataset_name}_bpp_vs_mse.png", dpi=150)
    plt.close()

    # 3. Entropy vs Achieved Compression
    plt.figure()
    plt.plot(jpeg_df['CR'], jpeg_df['Entropy'], marker='o', label='JPEG (DCT)', linewidth=2)
    plt.plot(j2k_df['CR'], j2k_df['Entropy'], marker='s', label='JPEG2000 (DWT)', linewidth=2)
    plt.title(f'Entropy vs Compression Ratio - {dataset_name}')
    plt.xlabel('Compression Ratio (CR)')
    plt.ylabel('Entropy (bits/symbol)')
    plt.legend()
    plt.grid(True, alpha=0.35)
    plt.tight_layout()
    plt.savefig(f"plots/{dataset_name}_entropy_vs_cr.png", dpi=150)
    plt.close()

    # 4. Energy Compaction Curves (transform coding)
    sample_path = pick_sample_image(dataset_name)
    if sample_path is not None:
        img = plt.imread(sample_path)
        gray = ensure_grayscale_float(img)
        dct_curve = cumulative_energy_curve(block_dct_coeffs(gray))
        dwt_curve = cumulative_energy_curve(dwt_coeffs(gray))

        plt.figure()
        plt.plot(dct_curve[0], dct_curve[1], label='JPEG transform (Block DCT)', linewidth=2)
        plt.plot(dwt_curve[0], dwt_curve[1], label='JPEG2000 transform (DWT)', linewidth=2)
        plt.title(f'Energy Compaction Curves - {dataset_name}')
        plt.xlabel('Fraction of Coefficients Kept')
        plt.ylabel('Cumulative Energy Retained')
        plt.ylim(0.0, 1.01)
        plt.xlim(0, 0.07)
        plt.legend(loc='lower right')
        plt.grid(True, alpha=0.35)
        plt.tight_layout()
        plt.savefig(f"plots/{dataset_name}_energy_compaction_curve.png", dpi=150)
        plt.close()